In [77]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd

In [78]:
# load data
SP500_df = pd.read_csv('SP500.csv')

# take log-diffs and drop nans
SP500_df['log_returns'] = SP500_df['SP500'].apply(lambda x: np.log(x)).diff()

# set index
SP500_df['observation_date'] = pd.DatetimeIndex(SP500_df['observation_date'])
SP500_df = SP500_df.set_index('observation_date')

# print dates
print(f'Data from {SP500_df.index[0]} to {SP500_df.index[-1]}')

# drop nans -> inspect which is nans
SP500_df = SP500_df.dropna()

Data from 2016-09-19 00:00:00 to 2026-09-17 00:00:00


In [ ]:
Delta = 1/252
T = len(SP500_df['log_returns'])
theta_init = SP500_df['log_returns'].var()/Delta


def eps_V(kappa, kappatheta, sigma_sq_V, Vt, Vtplus):

    num = Vtplus - Vt - kappatheta*Delta + kappa*Vt*Delta
    denom = np.sqrt(sigma_sq_V*Vt * Delta)

    return num/denom

def eps_Y(mu, Yt, Ytplus, Vt):

    num = Ytplus - Yt - mu*Delta + 0.5*Vt*Delta
    denom = np.sqrt(Vt*Delta)

    return num/denom

def obs_loglikelihood_t(mu, kappa, kappatheta, sigma_sq_V, rho, Yt, Ytplus, Vt, Vtplus):

    eps_V_t = eps_V(kappa, kappatheta, sigma_sq_V, Vt, Vtplus)

    #fac = 1/np.sqrt(Vt*(1-rho**2))
    fac = -0.5*np.log(2*np.pi*Vt*(1-rho**2)*Delta)

    exp_num = Ytplus - Yt - mu*Delta + 0.5*Vt*Delta - rho*np.sqrt(Vt*Delta)*eps_V_t
    exp_denom = 2*Vt*Delta*(1-rho**2)

    dens = fac - (exp_num**2)/exp_denom

    return dens

def latent_loglikelihood_t(kappa, kappatheta, sigma_sq_V, Vt, Vtplus):

    #fac = 1/np.sqrt(Vt*Delta*sigma_sq_V)
    fac = -0.5*np.log(2*np.pi*sigma_sq_V*Vt*Delta)

    exp_num = Vtplus - Vt - kappatheta*Delta+kappa*Vt*Delta
    exp_denom = 2*(sigma_sq_V)*Vt*Delta

    dens = fac - (exp_num**2)/exp_denom

    return dens

def all_obs_loglikelihoods(mu, kappa, kappatheta, sigma_sq_V, rho, allV):
    
    Y = np.array(SP500_df['log_returns'])[:-1] # T-1: Y1,Y2,...,YT-1
    Yplus = np.array(SP500_df['log_returns'])[1:] # T-1: Y2,Y3,...,YT
    V = allV[:-1] # T-1: V1,V2,...,VT-1
    Vplus = allV[1:] # T-1: V2,V3,...,VT

    obs_loglikelihoods = obs_loglikelihood_t(mu, kappa, kappatheta, sigma_sq_V, rho, Y, Yplus, V, Vplus)

    return np.sum(obs_loglikelihoods)

def all_latent_loglikelihoods(kappa, kappatheta, sigma_sq_V, allV):

    V = allV[:-1] # T-1: V1,V2,...,VT-1
    Vplus = allV[1:] # T-1: V2,V3,...,VT

    latent_loglikelihoods = latent_loglikelihood_t(kappa, kappatheta, sigma_sq_V, V, Vplus)

    return np.sum(latent_loglikelihoods)

np.float64(7199.10806527107)

In [198]:
def mu_step(priors, Theta_fill, V):

    mu_prior = priors['mu_mu']
    sigma_sq_prior = priors['sigma_sq_mu']

    # unpack other par
    kappa = Theta_fill['kappa']
    kappatheta = Theta_fill['kappatheta']
    sigma_sq_V = Theta_fill['sigma_sq_V']
    rho = Theta_fill['rho']

    # regression
    Yt = np.array(SP500_df['log_returns'])[:-1] # T-1: Y1,Y2,...,YT-1
    Ytplus = np.array(SP500_df['log_returns'].shift(-1))[:-1] # T-1: Y2,Y3,...,YT
    Vt = V[:-1] # T-1: V1,V2,...,VT-1
    Vtplus = V[1:] # T-1: V2,V3,...,VT
    eps_Vt = eps_V(kappa, kappatheta, sigma_sq_V, Vt, Vtplus)
    y = ((Ytplus - Yt + 0.5*Vt*Delta)/(np.sqrt(Vt*Delta)) - rho*eps_Vt)/(np.sqrt(1-rho**2))
    x = Delta/(np.sqrt(Vt*Delta*(1-rho**2)))

    sigma_sq_posterior = 1/((1/sigma_sq_prior) + np.sum(x**2))
    mu_posterior = sigma_sq_posterior*((mu_prior/sigma_sq_prior)+np.sum(x*y))

    # draw normal
    mu_new = np.random.normal(loc=mu_posterior, scale=np.sqrt(sigma_sq_posterior))

    return mu_new

def kappatheta_kappa_step(priors, Theta_fill, V):

    mu_kappa = priors['mu_kappa']
    sigma_sq_kappa = priors['sigma_sq_kappa']
    mu_kappatheta = priors['mu_kappatheta']
    sigma_sq_kappatheta = priors['sigma_sq_kappatheta']

    # unpack other par
    mu = Theta_fill['mu']
    sigma_sq_V = Theta_fill['sigma_V']
    rho = Theta_fill['rho']

    # construct variance-covariance matrix of prior
    Omega = np.diag([sigma_sq_kappatheta, sigma_sq_kappa])

    Mu = np.array([
        [mu_kappatheta],
        [mu_kappa]
    ])

    # regression
    Yt = np.array(SP500_df['log_returns'])[:-1] # T-1: Y1,Y2,...,YT-1
    Ytplus = np.array(SP500_df['log_returns'].shift(-1))[:-1] # T-1: Y2,Y3,...,YT
    Vt = V[:-1] # T-1: V1,V2,...,VT-1
    Vtplus = V[1:] # T-1: V2,V3,...,VT
    eps_Yt = eps_Y(mu, Yt, Ytplus, Vt)
    y = (((Vtplus - Vt)/(np.sqrt(sigma_sq_V*Vt*Delta))) - rho* eps_Yt)/(np.sqrt(1-rho**2))
    x1 = (np.sqrt(Delta))/(np.sqrt(sigma_sq_V*Vt*(1-rho**2)))
    x2 = - (np.sqrt(Vt*Delta))/(np.sqrt(sigma_sq_V*(1-rho**2)))
    X = np.concatenate([x1[:, None], x2[:, None]], axis=1)
    Omega_posterior = np.linalg.inv(np.linalg.inv(Omega)+X.T @ X)
    Mu_posterior = Omega_posterior @ (np.linalg.inv(Omega) @ Mu + (X.T @ y)[:,None])

    draws = np.random.multivariate_normal(mean=Mu_posterior.flatten(), cov=Omega_posterior)
    kappatheta, kappa = draws[0].item(), draws[1].item()

    return kappatheta, kappa

def sigma_V_step(priors, Theta, Theta_fill, V):

    alpha_sigma_sq_V = priors['alpha_sigma_sq_V']
    beta_sigma_sq_V = priors['beta_sigma_sq_V']

    # unpack other par
    mu = Theta_fill['mu']
    kappa = Theta_fill['kappa']
    kappatheta = Theta_fill['kappatheta']
    rho = Theta_fill['rho']

    # obtain proposal draw by bayesian regression
    Vt = V[:-1] # T-1: V1,V2,...,VT-1
    Vtplus = V[1:] # T-1: V2,V3,...,VT
    y = (Vtplus - Vt - kappatheta*Delta + kappa*Vt*Delta)/(np.sqrt(Vt*Delta)) # T-1

    alpha_sigma_sq_V_post = alpha_sigma_sq_V + (T-1)/2
    beta_sigma_sq_V_post = beta_sigma_sq_V + (np.sum(y**2))/2

    sigma_sq_V_proposal = stats.invgamma.rvs(alpha_sigma_sq_V_post, scale=beta_sigma_sq_V_post)

    # unpack old MH-draw
    sigma_sq_V_g = Theta['sigma_sq_V']

    # do MH-step
    # posterior log density for gth draw
    obs_ll_g = all_obs_loglikelihoods(mu, kappa, kappatheta, sigma_sq_V_g, rho, V)
    latent_ll_g = all_latent_loglikelihoods(kappa, kappatheta, sigma_sq_V_g, V)
    prior_g = stats.invgamma.logpdf(sigma_sq_V_g, alpha_sigma_sq_V, scale=beta_sigma_sq_V)
    post_g = obs_ll_g + latent_ll_g + prior_g

    # posterior log density for proposal draw
    obs_ll_proposal = all_obs_loglikelihoods(mu, kappa, kappatheta, sigma_sq_V_proposal, rho, V)
    latent_ll_proposal = all_latent_loglikelihoods(kappa, kappatheta, sigma_sq_V_proposal, V)
    prior_proposal = stats.invgamma.logpdf(sigma_sq_V_proposal, alpha_sigma_sq_V, scale=beta_sigma_sq_V)
    post_proposal = obs_ll_proposal + latent_ll_proposal + prior_proposal

    # proposal density for gth draw
    proposal_dens_g = stats.invgamma.logpdf(sigma_sq_V_g, alpha_sigma_sq_V_post, scale=beta_sigma_sq_V_post)

    # proposal density for proposal
    proposal_dens_proposal = stats.invgamma.logpdf(sigma_sq_V_proposal, alpha_sigma_sq_V_post, scale=beta_sigma_sq_V_post)

    # compute log(alpha)
    log_alpha = post_proposal - post_g + proposal_dens_g - proposal_dens_proposal

    # accept if log(alpha) > log(u)
    u = np.random.uniform()
    logu = np.log(u)

    if log_alpha > logu:
        sigma_sq_V_gplus = sigma_sq_V_proposal

    else:
        sigma_sq_V_gplus = sigma_sq_V_g

    return sigma_sq_V_gplus

def rho_step(priors, Theta, Theta_fill, V):

    # unpack other par
    mu = Theta_fill['mu']
    kappa = Theta_fill['kappa']
    kappatheta = Theta_fill['kappatheta']
    sigma_sq_V = Theta_fill['sigma_sq_V']

    # obtain proposal
    Yt = np.array(SP500_df['log_returns'])[:-1] # T-1: Y1,Y2,...,YT-1
    Ytplus = np.array(SP500_df['log_returns'].shift(-1))[:-1] # T-1: Y2,Y3,...,YT
    Vt = V[:-1] # T-1: V1,V2,...,VT-1
    Vtplus = V[1:] # T-1: V2,V3,...,VT
    eps_Vt = eps_V(kappa, kappatheta, sigma_sq_V, Vt, Vtplus)
    eps_Yt = eps_Y(mu, Yt, Ytplus, Vt)

    rho_mean = np.corrcoef(eps_Vt, eps_Yt)[0,1].item()
    rho_std = ((1-rho_mean**2)/np.sqrt(T-1)).item()

    rho_proposal = np.clip(np.random.normal(loc=rho_mean, scale=rho_std), -1, 1).item()

    # unpack old MH-draw
    rho_g = Theta['rho']

    # do MH-step, prior is uninformative
    # posterior log density for gth draw, latent likelihood is unaffected by rho
    obs_ll_g = all_obs_loglikelihoods(mu, kappa, kappatheta, sigma_sq_V, rho_g, V)
    post_g = obs_ll_g

    # posterior log density for proposal draw
    obs_ll_proposal = all_obs_loglikelihoods(mu, kappa, kappatheta, sigma_sq_V, rho_proposal, V)
    post_proposal = obs_ll_proposal

    # proposal density for gth draw
    proposal_dens_g = stats.norm.logpdf(rho_g, loc=rho_mean, scale=rho_std)

    # proposal density for proposal draw
    proposal_dens_proposal = stats.norm.logpdf(rho_proposal, loc=rho_mean, scale=rho_std)

    # compute log(alpha)
    log_alpha = post_proposal - post_g + proposal_dens_g - proposal_dens_proposal

    # accept if log(alpha) > log(u)
    u = np.random.uniform()
    logu = np.log(u)

    if log_alpha > logu:
        rho_gplus = rho_proposal

    else:
        rho_gplus = rho_g

    return rho_gplus

rho_step(priors, init_guess, init_guess, np.ones(T))

-0.011221519505726826

In [163]:
init_guess = {
    'mu' : 0.00,
    'kappa' : 1.0,
    'kappatheta' : theta_init,
    'sigma_sq_V' : 0.001,
    'rho' : -0.5,
    'V' : np.ones(T)*theta_init
}

priors = {

    # mu
    'mu_mu' : 0.0,
    'sigma_sq_mu' : 1.0,

    # kappa
    'mu_kappa' : 0.0,
    'sigma_sq_kappa' : 1.0,

    # kappatheta
    'mu_kappatheta' : 0.0,
    'sigma_sq_kappatheta' : 1.0,

    # sigma_V
    'alpha_sigma_sq_V' : 2.5,
    'beta_sigma_sq_V' : 0.1,

    # rho
    'rho_low' : -1,
    'rho_high' : 1
    
}

In [ ]:
def cycle(Theta_g, V_g):

    # unpack
    mu = Theta_g['mu']
    kappa = Theta_g['kappa']
    kappatheta = Theta_g['kappatheta']
    sigma_V = Theta_g['sigma_V']
    rho = Theta_g['rho']

    # allocate
    Theta_fill = Theta_g.copy()

    # mu
    mu_gplus = mu_step(priors, Theta_fill, V_g)
    Theta_fill['mu'] = mu_gplus

    # kappa, kappatheta
    kappatheta_gplus, kappa_gplus = kappatheta_kappa_step(priors, Theta_fill, V_g)
    Theta_fill['kappatheta'] = kappatheta_gplus
    Theta_fill['kappa'] = kappa_gplus

    # sigma_sq_V
    sigma_sq_V_gplus = sigma_V_step(priors, Theta_g, Theta_fill, V_g)
    Theta_fill['sigma_sq_V'] = sigma_sq_V_gplus

    # rho
    rho_gplus = rho_step(priors, Theta_g, Theta_fill, V_g)
    Theta_fill['rho'] = rho_gplus